# Dataset Analysis: circRNA–miRNA and lncRNA–miRNA interaction data
**Purpose:** before building any model, answer: *what data do we have, how big is it, how biased is it, and can the datasets be merged?*

Sources (all from your uploads):
| Key | Source | RNA type |
|---|---|---|
| `HMA_D1`, `HMA_D2`, `HMA_D3` | HMA-GCA `datasets/dataset{1,2,3}.7z` | circRNA |
| `DeepLMI` | DeepLMI `dataset/` (+ `dataset_in/` independent set, `cold_data_splits/`) | lncRNA |
| `LMIMHGAT_L5`, `LMIMHGAT_main` | LMI-MHGAT `xiao/data` (layer 5 of `extra.txt`, and `main.txt`) | lncRNA |

Sections: 0 setup · 1 inventory · 2 loaders · 3 size and sparsity · 4 degree bias · 5 popularity baseline · 6 features and sequences · 7 miRNA overlap · 8 merged table · 9 summary.

In [ ]:
import os
base = "/kaggle/input/datasets/jasminsultanashimu/combine-dataset/combine-dataset"
print(os.listdir(base))   # expect ['deep-lmi-dataset', 'hma-gca-dataset', ...]
print(os.listdir(base + "/hma-gca-dataset/dataset1"))  # also shows the matrix filename

In [ ]:
import os, re, glob, zipfile, subprocess, warnings
import numpy as np, pandas as pd
import matplotlib; 
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display = print
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# ---------------- CONFIG: edit these ----------------
UPLOADS  = "/kaggle/input/datasets/jasminsultanashimu/combine-dataset/combine-dataset"
ONLY_ZIP = None
EXTRACT  = "/kaggle/input/datasets/jasminsultanashimu/combine-dataset/combine-dataset"
OUT      = "./analysis_output"

SEED = 42
# ----------------------------------------------------

os.makedirs(EXTRACT, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

rng = np.random.default_rng(SEED)

## 0. Unpack the archives
Zip files are unpacked first, then any `.7z` inside (the HMA-GCA datasets). Big irrelevant files (RNA contact maps, model checkpoints, venvs) are skipped.
If the `.7z` step fails, extract the three `dataset*.7z` files manually into `./extracted/` and re-run the next cell.

In [ ]:
SKIP = ("/mirna_f/", "prob_single", ".pt", ".pth", "/venv/", "/.idea/", "__pycache__", "/mirna_contact/", "/merged_mirna_contact/", "/mirna_emb/", "/merged_mirna_emb/")
def extract_zips(src, dst):
    for z in glob.glob(os.path.join(src, "**", "*.zip"), recursive=True):
        if ONLY_ZIP and os.path.basename(z) != ONLY_ZIP: continue
        tgt = os.path.join(dst, os.path.splitext(os.path.basename(z))[0])
        if os.path.exists(tgt): continue
        with zipfile.ZipFile(z) as f:
            for m in f.namelist():
                if any(s in "/"+m for s in SKIP) or m.endswith("/"): continue
                f.extract(m, tgt)
        print("unzipped", z)

def extract_7z(dst):
    import sys, shutil
    for s in glob.glob(os.path.join(dst, "**", "*.7z"), recursive=True):
        tgt = s[:-3] + "_x"
        if os.path.exists(tgt) and os.listdir(tgt): continue
        os.makedirs(tgt, exist_ok=True); ok = False
        try:
            try: import py7zr
            except ImportError:
                for extra in ([], ["--break-system-packages"]):
                    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py7zr"] + extra, capture_output=True).returncode == 0: break
                import py7zr
            with py7zr.SevenZipFile(s) as f: f.extractall(tgt)
            ok = True
        except Exception as e:
            err = e
        if not ok:
            try: ok = subprocess.run(["7z", "x", "-y", f"-o{tgt}", s], capture_output=True).returncode == 0
            except FileNotFoundError: ok = False
        if ok: print("7z extracted", s)
        else:
            shutil.rmtree(tgt, ignore_errors=True)
            print("!! could not extract", s, "-> extract it manually into", dst, "(needs py7zr via pip, or the 7z program)")

extract_zips(UPLOADS, EXTRACT); extract_7z(EXTRACT)

## 1. File inventory
Everything that was unpacked, so you can confirm the right folders are being used.

In [ ]:
rows=[]
for p in glob.glob(os.path.join(EXTRACT, "**", "*"), recursive=True):
    if os.path.isfile(p):
        rows.append((os.path.relpath(p, EXTRACT), round(os.path.getsize(p)/1024,1)))
inv = pd.DataFrame(rows, columns=["file","KB"])
inv["top_folder"] = inv.file.str.split(os.sep).str[0]
print(len(inv), "files")
display(inv.groupby("top_folder").agg(files=("file","count"), MB=("KB", lambda x: round(x.sum()/1024,2))))
hma_hits = sorted(glob.glob(os.path.join(EXTRACT, "**", "circRNA_kmer_dict.txt"), recursive=True))
print("HMA-GCA dataset folders found:"); [print("  ", os.path.dirname(h)) for h in hma_hits]

## 2. Loaders
Everything is converted to one common object: a binary **interaction matrix `A` (rows = miRNAs, columns = RNAs)** plus IDs, so all datasets can be analysed with the same code.
HMA-GCA ID order is taken from the first column of its feature files (the same order as its adjacency matrix, as used in `main.py`).

In [ ]:
class DS:
    def __init__(s, name, rna_type, rna_ids, mi_ids, A, **extra):
        s.name, s.rna_type, s.rna_ids, s.mi_ids, s.A = name, rna_type, list(rna_ids), list(mi_ids), A.astype(np.int8)
        s.extra = extra
    @property
    def pairs(s): return np.argwhere(s.A == 1)          # (mi_idx, rna_idx)

def read_ids(path):
    with open(path) as f: return [l.split(",")[0].strip() for l in f if l.strip()]
def read_desc(path):
    with open(path) as f: return np.array([[float(x) for x in l.strip().split(",")[1:]] for l in f if l.strip()], dtype=np.float32)

def load_hma(d, name):
    P = lambda n: os.path.join(d, n)
    circ = read_ids(P("circRNA_kmer_dict.txt")); mi = read_ids(P("miRNA_kmer_dict.txt"))
    mf = [f for f in os.listdir(d) if "matrix" in f.lower()][0]
    A = pd.read_csv(P(mf), header=None).values
    if A.shape == (len(circ), len(mi)) and A.shape != (len(mi), len(circ)): A = A.T; print(name, ": matrix transposed to (miRNA x circRNA)")
    assert A.shape == (len(mi), len(circ)), f"{name}: matrix {A.shape} vs ids {len(mi)}x{len(circ)}"
    feats = {}
    for typ in ["circRNA", "miRNA"]:
        for k in ["kmer", "ctd", "doc2vec", "role2vec"]:
            fp = P(f"{typ}_{k}_dict.txt")
            if os.path.exists(fp): feats[(typ, k)] = read_desc(fp)
    return DS(name, "circRNA", circ, mi, A, feats=feats, files=sorted(os.listdir(d)))

def read_fasta(path):
    out, k = {}, None
    for l in open(path):
        l = l.strip()
        if l.startswith(">"): k = l[1:]; out[k] = ""
        elif k: out[k] += l
    return out

def load_deeplmi(csv_path):
    base = os.path.dirname(csv_path)
    d = pd.read_csv(csv_path)
    lnc = sorted(d.lncrna.unique()); mi = sorted(d.mirna.unique())
    li = {x:i for i,x in enumerate(lnc)}; mj = {x:i for i,x in enumerate(mi)}
    A = np.zeros((len(mi), len(lnc)), dtype=np.int8)
    for a,b in zip(d.lncrna, d.mirna): A[mj[b], li[a]] = 1
    ds = DS("DeepLMI", "lncRNA", lnc, mi, A, table=d,
            fa_lnc=read_fasta(os.path.join(base,"lncrna.fasta")), fa_mi=read_fasta(os.path.join(base,"mirna.fasta")))
    ds_in = None
    # optional: independent set (indices -> names via index_value.csv), only if present
    cand = glob.glob(os.path.join(EXTRACT, "**", "dataset_in", "node_link.csv"), recursive=True)
    if cand:
        di = os.path.dirname(cand[0])
        iv = pd.read_csv(os.path.join(di, "index_value.csv")); idx2n = dict(zip(iv["index"], iv["rna"]))
        fa_mi_in = read_fasta(os.path.join(di, "mirna.fasta")); fa_l_in = read_fasta(os.path.join(di, "lncrna.fasta"))
        nl = pd.read_csv(os.path.join(di, "node_link.csv")); prs=[]
        for a,b in zip(nl.node1, nl.node2):
            a, b = idx2n[a], idx2n[b]
            if a in fa_mi_in and b in fa_l_in: prs.append((b,a))
            elif b in fa_mi_in and a in fa_l_in: prs.append((a,b))
        ind = pd.DataFrame(prs, columns=["lncrna","mirna"]).drop_duplicates()
        lnc2 = sorted(ind.lncrna.unique()); mi2 = sorted(ind.mirna.unique()); li2 = {x:i for i,x in enumerate(lnc2)}; mj2 = {x:i for i,x in enumerate(mi2)}
        A2 = np.zeros((len(mi2), len(lnc2)), dtype=np.int8)
        for a,b in zip(ind.lncrna, ind.mirna): A2[mj2[b], li2[a]] = 1
        ds_in = DS("DeepLMI_independent", "lncRNA", lnc2, mi2, A2, fa_lnc=fa_l_in, fa_mi=fa_mi_in)
    # optional: cold-start splits
    cold=[]
    for f in sorted(glob.glob(os.path.join(EXTRACT, "**", "cold_data_splits", "*_train_*.csv"), recursive=True)):
        tr = pd.read_csv(f); va = pd.read_csv(f.replace("_train_", "_val_"))
        b = os.path.basename(f).split("_train_")
        cold.append((b[0], int(b[1][0]), len(tr), int(tr.label.sum()), len(va), int(va.label.sum())))
    ds.extra["cold"] = pd.DataFrame(cold, columns=["setting","fold","n_train","pos_train","n_val","pos_val"])
    return ds, ds_in

def load_lmimhgat(D):
    ex = pd.read_csv(os.path.join(D, "extra.txt"), sep=r"\s+", header=None, names=["layer","mirna","rna"])
    mn = pd.read_csv(os.path.join(D, "main.txt"),  sep=r"\s+", header=None, names=["layer","mirna","rna"])
    nt = pd.read_csv(os.path.join(D, "node_type.txt"), sep=r"\s+", header=None, names=["id","type"])
    def mk(df, name):
        r = sorted(df.rna.unique()); m = sorted(df.mirna.unique()); ri={x:i for i,x in enumerate(r)}; mi={x:i for i,x in enumerate(m)}
        A = np.zeros((len(m), len(r)), dtype=np.int8)
        for a,b in zip(df.rna, df.mirna): A[mi[b], ri[a]] = 1
        return DS(name, "lncRNA", r, m, A)
    L5 = ex[ex.layer == 5]
    return mk(L5, "LMIMHGAT_L5"), mk(mn.drop_duplicates(["mirna","rna"]), "LMIMHGAT_main"), ex.layer.value_counts().sort_index(), nt.type.value_counts()

In [ ]:
datasets = {}
for h in hma_hits:
    d = os.path.dirname(h); m = re.findall(r"dataset\s*_?(\d)", d.lower())
    key = f"HMA_D{m[-1]}" if m else "HMA_" + os.path.basename(d)
    datasets[key] = load_hma(d, key)

dl_hit = glob.glob(os.path.join(EXTRACT, "**", "mirna_lncrna_interaction.csv"), recursive=True)
dl_root = dl_hit[0] if dl_hit else None
if dl_root:
    datasets["DeepLMI"], ds_in = load_deeplmi(dl_root)
    if ds_in is not None: datasets["DeepLMI_independent"] = ds_in
lm_hit = [p for p in glob.glob(os.path.join(EXTRACT, "**", "extra.txt"), recursive=True) if os.path.exists(os.path.join(os.path.dirname(p), "main.txt"))]
lm_root = os.path.dirname(lm_hit[0]) if lm_hit else None
if lm_root:
    L5, LM, layers, ntypes = load_lmimhgat(lm_root)
    datasets["LMIMHGAT_L5"], datasets["LMIMHGAT_main"] = L5, LM
datasets = dict(sorted(datasets.items()))
print("Loaded:", list(datasets))
if not any(k.startswith("HMA") for k in datasets):
    print("!! No HMA-GCA dataset found. Extract dataset{1,2,3}.7z into", EXTRACT, "and re-run this cell.")

### Check against the numbers published in the papers
If a loaded count differs from the paper, the file or the paper has an inconsistency - worth knowing before any experiment.

In [ ]:
paper = {"HMA_D1":(2115,821,9589),"HMA_D2":(2346,926,9905),"HMA_D3":(3569,1152,20208),"LMIMHGAT_L5":(908,276,7698)}
rows=[]
for k,(r,m,p) in paper.items():
    if k in datasets:
        d=datasets[k]; rows.append(dict(dataset=k,paper_RNAs=r,file_RNAs=d.A.shape[1],paper_miRNAs=m,file_miRNAs=d.A.shape[0],paper_pairs=p,file_pairs=int(d.A.sum()),
                                      match=(r,m,p)==(d.A.shape[1],d.A.shape[0],int(d.A.sum()))))
display(pd.DataFrame(rows))

## 3. Size and sparsity
How many RNAs, miRNAs and known pairs, and how sparse the interaction matrix is (density = known pairs ÷ all possible pairs).

In [ ]:
def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return 0.0 if x.sum()==0 else (2*np.sum((np.arange(1,n+1))*x)/(n*x.sum()) - (n+1)/n)
def top_share(deg, frac=0.1):
    k = max(1,int(round(len(deg)*frac))); return np.sort(deg)[::-1][:k].sum()/deg.sum()

def stats(ds):
    A=ds.A; dr=A.sum(0); dm=A.sum(1); P=int(A.sum()); nm,nr=A.shape
    return dict(dataset=ds.name, rna_type=ds.rna_type, n_RNA=nr, n_miRNA=nm, positives=P, possible_pairs=nm*nr,
        density_pct=round(100*P/(nm*nr),4), neg_per_pos_if_all_unlabeled=round((nm*nr-P)/P,1),
        RNA_deg_mean=round(dr.mean(),2), RNA_deg_median=float(np.median(dr)), RNA_deg_max=int(dr.max()), RNA_deg1_pct=round(100*(dr==1).mean(),1),
        miRNA_deg_mean=round(dm.mean(),2), miRNA_deg_median=float(np.median(dm)), miRNA_deg_max=int(dm.max()), miRNA_deg1_pct=round(100*(dm==1).mean(),1),
        RNA_gini=round(gini(dr),3), miRNA_gini=round(gini(dm),3),
        top10pct_RNA_edge_share=round(top_share(dr),3), top10pct_miRNA_edge_share=round(top_share(dm),3))
S = pd.DataFrame([stats(d) for d in datasets.values()]).set_index("dataset")
display(S.T); S.to_csv(f"{OUT}/dataset_summary.csv")

fig, ax = plt.subplots(1,3, figsize=(15,4))
S[["n_RNA","n_miRNA"]].plot.bar(ax=ax[0]); ax[0].set_title("Nodes per dataset"); ax[0].set_yscale("log")
S["positives"].plot.bar(ax=ax[1], color="teal"); ax[1].set_title("Known interactions (positives)")
S["density_pct"].plot.bar(ax=ax[2], color="indianred"); ax[2].set_title("Matrix density (%)"); ax[2].set_yscale("log")
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_sizes.png"); plt.show()

In [ ]:
if dl_root and len(datasets["DeepLMI"].extra["cold"]):
    print("DeepLMI cold-start splits (per fold sizes):")
    display(datasets["DeepLMI"].extra["cold"].groupby("setting").agg(folds=("fold","count"), train_mean=("n_train","mean"), val_mean=("n_val","mean"), pos_frac_train=("pos_train", lambda x: np.nan)).drop(columns="pos_frac_train"))
if lm_root:
    print("LMI-MHGAT interaction layers in extra.txt (layer 5 = lncRNA-miRNA used as target):"); display(layers.to_frame("edges").T)
    print("LMI-MHGAT node types:"); display(ntypes.to_frame("nodes").T)

## 4. Degree distributions and popularity bias
If a few RNAs/miRNAs hold most of the edges, a model can score well just by predicting "popular nodes interact". These plots show how strong that effect is in each dataset.

In [ ]:
def ccdf(x):
    x=np.sort(np.asarray(x)); y=1-np.arange(len(x))/len(x); return x,y
n=len(datasets); cols=min(4,n); rows=int(np.ceil(n/cols))
fig,axs=plt.subplots(rows,cols,figsize=(4.2*cols,3.6*rows),squeeze=False)
for ax,(k,ds) in zip(axs.ravel(),datasets.items()):
    for deg,lab,c in [(ds.A.sum(0),f"{ds.rna_type}","tab:blue"),(ds.A.sum(1),"miRNA","tab:orange")]:
        x,y=ccdf(deg[deg>0]); ax.loglog(x,y,label=lab,color=c)
    ax.set_title(k); ax.set_xlabel("degree"); ax.set_ylabel("P(deg ≥ x)"); ax.legend()
for ax in axs.ravel()[n:]: ax.axis("off")
plt.suptitle("Degree CCDF (log-log): straight line = heavy tail", y=1.02); plt.tight_layout(); plt.savefig(f"{OUT}/fig_degree_ccdf.png",bbox_inches="tight"); plt.show()

fig,axs=plt.subplots(1,2,figsize=(11,4))
for k,ds in datasets.items():
    for ax,deg,t in [(axs[0],ds.A.sum(0),"RNA side"),(axs[1],ds.A.sum(1),"miRNA side")]:
        d=np.sort(deg)[::-1]; cs=np.cumsum(d)/d.sum(); ax.plot(np.arange(1,len(d)+1)/len(d),cs,label=k); ax.set_title(f"Lorenz-style: share of edges held by top x% ({t})")
for ax in axs: ax.plot([0,1],[0,1],"k:"); ax.set_xlabel("fraction of nodes (most connected first)"); ax.set_ylabel("fraction of edges")
axs[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/fig_lorenz.png"); plt.show()

In [ ]:
def block_heat(A, maxn=250):
    A=A[np.argsort(-A.sum(1))][:,np.argsort(-A.sum(0))].astype(float)
    ri=np.linspace(0,A.shape[0],min(maxn,A.shape[0])+1).astype(int); ci=np.linspace(0,A.shape[1],min(maxn,A.shape[1])+1).astype(int)
    return np.add.reduceat(np.add.reduceat(A,ri[:-1],axis=0),ci[:-1],axis=1)
fig,axs=plt.subplots(rows,cols,figsize=(4.2*cols,3.8*rows),squeeze=False)
for ax,(k,ds) in zip(axs.ravel(),datasets.items()):
    ax.imshow(np.log1p(block_heat(ds.A)),aspect="auto",cmap="viridis"); ax.set_title(k); ax.set_xlabel(f"{ds.rna_type} (sorted by degree)"); ax.set_ylabel("miRNA (sorted)")
for ax in axs.ravel()[n:]: ax.axis("off")
plt.suptitle("Interaction matrix sorted by degree: bright top-left corner = hubs dominate",y=1.02); plt.tight_layout(); plt.savefig(f"{OUT}/fig_matrix_heat.png",bbox_inches="tight"); plt.show()

## 5. Popularity baseline (no learning, no sequences)
Score a pair as *(RNA degree) × (miRNA degree)* using only the training fold, then measure AUC under 5-fold CV.
- **Random negatives** (what HMA-GCA does): negatives are uniform random unlabeled pairs.
- **Degree-matched negatives**: negatives are sampled with the same degree bias as the positives, so the popularity shortcut should disappear.

**Note:** with degree-matched negatives the AUC can land slightly *below* 0.5. That is expected in small sparse graphs: hiding a test positive lowers its own endpoints' training degree, while negatives are not affected. Read it as "popularity no longer helps", not as a real signal.

**How to read it:** if the random-negative AUC is far above 0.5, a published AUC of ~0.95 is partly just popularity. The drop under degree-matched negatives measures the shortcut.

In [ ]:
from sklearn.model_selection import KFold
def sample_negs(A, At, n, mode, rng):
    # negatives = unlabeled pairs (A==0). 'degree' mode draws RNA and miRNA with probability proportional to their full-graph degree (same marginals as the positives),
    # so negatives look as popular as positives (no shortcut).
    nm, nr = A.shape; negs = set()
    if mode == "degree":
        pr = A.sum(0) + 1e-9; pm = A.sum(1) + 1e-9; pr = pr/pr.sum(); pm = pm/pm.sum()
    for _ in range(500):
        if len(negs) >= n: break
        k = (n - len(negs)) * 3
        if mode == "degree": i = rng.choice(nm, k, p=pm); j = rng.choice(nr, k, p=pr)
        else: i = rng.integers(0, nm, k); j = rng.integers(0, nr, k)
        for a_, b_ in zip(i, j):
            if A[a_, b_] == 0: negs.add((a_, b_))
            if len(negs) >= n: break
    return np.array(list(negs))

def pop_baseline(ds, mode, seed=SEED, folds=5):
    rng = np.random.default_rng(seed); A = ds.A; pos = ds.pairs
    res = {"both": [], "RNA_only": [], "miRNA_only": []}; ap = []
    for tr, te in KFold(folds, shuffle=True, random_state=seed).split(pos):
        At = A.copy()
        for m, r in pos[te]: At[m, r] = 0                      # hide test positives from the graph
        neg = sample_negs(A, At, len(te), mode, rng)
        X = np.vstack([pos[te], neg]); y = np.r_[np.ones(len(te)), np.zeros(len(neg))]
        dr = At.sum(0); dm = At.sum(1); m, r = X[:, 0], X[:, 1]
        res["both"].append(roc_auc_score(y, dm[m]*dr[r])); ap.append(average_precision_score(y, dm[m]*dr[r]))
        res["RNA_only"].append(roc_auc_score(y, dr[r])); res["miRNA_only"].append(roc_auc_score(y, dm[m]))
    return {**{f"AUC_{k}": np.mean(v) for k, v in res.items()}, "AUPR_both": np.mean(ap)}

rows=[]
for k,ds in datasets.items():
    for mode in ["random","degree"]:
        rows.append(dict(dataset=k,negatives=mode,**pop_baseline(ds,mode)))
PB=pd.DataFrame(rows).round(4); display(PB.pivot(index="dataset",columns="negatives",values="AUC_both").rename(columns=lambda c:f"AUC popularity ({c} negs)"))
display(PB); PB.to_csv(f"{OUT}/popularity_baseline.csv",index=False)
piv=PB.pivot(index="dataset",columns="negatives",values="AUC_both"); ax=piv.plot.bar(figsize=(9,4)); ax.axhline(0.5,color="k",ls=":"); ax.set_ylabel("AUC (degree product, no learning)"); ax.set_title("Popularity baseline"); plt.tight_layout(); plt.savefig(f"{OUT}/fig_popularity.png"); plt.show()

## 6. Features and sequences
- **HMA-GCA** ships pre-computed descriptors (K-mer, CTD, Doc2Vec, Role2Vec, similarity), not raw sequences. We show their shapes and a 2-D PCA of the K-mer space coloured by degree.
- **DeepLMI** has raw sequences, so length and GC content are analysed directly.

In [ ]:
frows=[]
for k,ds in datasets.items():
    for (typ,f),M in ds.extra.get("feats",{}).items(): frows.append((k,typ,f,M.shape[0],M.shape[1],round(float((M==0).mean())*100,1)))
if frows: display(pd.DataFrame(frows,columns=["dataset","side","feature","rows","dims","pct_zeros"]))

hk=[k for k,d in datasets.items() if "feats" in d.extra]
if hk:
    fig,axs=plt.subplots(2,len(hk),figsize=(4.6*len(hk),7),squeeze=False)
    for c,k in enumerate(hk):
        ds=datasets[k]
        for r,(typ,deg) in enumerate([("circRNA",ds.A.sum(0)),("miRNA",ds.A.sum(1))]):
            M=ds.extra["feats"].get((typ,"kmer"))
            if M is None: continue
            Z=PCA(2,random_state=SEED).fit_transform((M-M.mean(0))/(M.std(0)+1e-6))
            sc=axs[r,c].scatter(Z[:,0],Z[:,1],c=np.log1p(deg),s=6,cmap="magma"); axs[r,c].set_title(f"{k} {typ} K-mer PCA"); plt.colorbar(sc,ax=axs[r,c],label="log(1+degree)")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_kmer_pca.png"); plt.show()

In [ ]:
if dl_root:
    def seqdf(fa,typ): 
        return pd.DataFrame({"id":list(fa),"len":[len(s) for s in fa.values()],"GC":[ (s.upper().count("G")+s.upper().count("C"))/max(len(s),1) for s in fa.values()],"type":typ})
    parts_=[seqdf(datasets["DeepLMI"].extra["fa_lnc"],"lncRNA (DeepLMI)"),seqdf(datasets["DeepLMI"].extra["fa_mi"],"miRNA (DeepLMI)")]
    if "DeepLMI_independent" in datasets:
        parts_+=[seqdf(datasets["DeepLMI_independent"].extra["fa_lnc"],"lncRNA (independent)"),seqdf(datasets["DeepLMI_independent"].extra["fa_mi"],"miRNA (independent)")]
    D=pd.concat(parts_)
    display(D.groupby("type")[["len","GC"]].describe().round(2).T)
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    for t,g in D[D.type.str.startswith("lncRNA")].groupby("type"): ax[0].hist(np.log10(g.len),bins=30,alpha=.6,label=t)
    ax[0].set_xlabel("log10 length (nt)"); ax[0].set_title("lncRNA lengths"); ax[0].legend()
    for t,g in D[D.type.str.startswith("miRNA")].groupby("type"): ax[1].hist(g.len,bins=range(15,35),alpha=.6,label=t)
    ax[1].set_xlabel("length (nt)"); ax[1].set_title("miRNA lengths"); ax[1].legend()
    sns.boxplot(data=D,x="type",y="GC",ax=ax[2]); ax[2].tick_params(axis="x",rotation=30); ax[2].set_title("GC content")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_sequences.png"); plt.show()
    t=datasets["DeepLMI"].extra["table"]; print(f"{t.mirna_seq.str.contains(',').sum()} of {len(t)} rows have 2+ miRNA sequences (precursor arms) in one cell - pick one arm when featurising.")
    print("lncRNAs longer than 1,022 nt (RNA-FM input limit):", int((D[D.type=='lncRNA (DeepLMI)'].len>1022).sum()), "of", int((D.type=='lncRNA (DeepLMI)').sum()))

## 7. miRNA harmonisation and overlap (the merge feasibility test)
Names differ across sources (`miR-491` vs `hsa-miR-124-3p`). Two normalisation levels:
- **exact**: lower-case, species prefix removed (`hsa-miR-21-5p` → `mir-21-5p`)
- **family (approximate)**: also drops `-3p/-5p`, paralog suffix and letter (`mir-125b-1-5p` → `mir-125`). Family matches are only a *loose* match: different family members can have different targets.

In [ ]:
def norm_mirna(x):
    x=str(x).strip().lower(); x=re.sub(r"^(hsa|mmu|rno)[-_]","",x); x=x.replace("microrna","mir"); x=re.sub(r"^mir(?=\d)","mir-",x); return x
def fam_mirna(x):
    n=norm_mirna(x); n=re.sub(r"-(3p|5p)$","",n); n=re.sub(r"(?<=[a-z])-\d$","",n); n=re.sub(r"(?<=\d)[a-z]$","",n); return n

mi_exact={k:{norm_mirna(m) for m in d.mi_ids} for k,d in datasets.items()}
mi_fam  ={k:{fam_mirna(m)  for m in d.mi_ids} for k,d in datasets.items()}
def overlap(sets):
    ks=list(sets); n=pd.DataFrame(index=ks,columns=ks,dtype=float); j=n.copy()
    for a in ks:
        for b in ks:
            i=len(sets[a]&sets[b]); n.loc[a,b]=i; j.loc[a,b]=i/len(sets[a]|sets[b])
    return n,j
fig,axs=plt.subplots(1,2,figsize=(15,5.5))
for ax,(t,s) in zip(axs,[("exact",mi_exact),("family (approx.)",mi_fam)]):
    nn,jj=overlap(s); sns.heatmap(nn,annot=True,fmt=".0f",cmap="Blues",ax=ax,cbar=False)
    ax.set_title(f"Shared miRNAs ({t}); diagonal = size"); display(jj.round(2).style.set_caption(f"Jaccard ({t})") if hasattr(jj,'style') else jj)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_mirna_overlap.png"); plt.show()

circ=[k for k,d in datasets.items() if d.rna_type=="circRNA"]; lnc=[k for k,d in datasets.items() if d.rna_type=="lncRNA" and "independent" not in k]
if circ and lnc:
    cu=set().union(*[mi_exact[k] for k in circ]); 
    print(f"circRNA-side miRNAs (union of {circ}): {len(cu)}")
    for k in lnc:
        sh=cu&mi_exact[k]; shf=set().union(*[mi_fam[c] for c in circ])&mi_fam[k]
        print(f"  {k}: {len(mi_exact[k])} miRNAs | exact shared: {len(sh)} ({100*len(sh)/len(mi_exact[k]):.0f}%) | family shared: {len(shf)} ({100*len(shf)/len(mi_fam[k]):.0f}%)")

In [ ]:
# Pair-level overlap between the circRNA datasets (needed to avoid train/test leakage in cross-dataset tests)
def pair_set(ds): return {(ds.rna_ids[r].lower(), norm_mirna(ds.mi_ids[m])) for m,r in ds.pairs}
if len(circ)>1:
    ps={k:pair_set(datasets[k]) for k in circ}; rs={k:{x.lower() for x in datasets[k].rna_ids} for k in circ}
    t=pd.DataFrame([[a,b,len(rs[a]&rs[b]),len(ps[a]&ps[b]),round(100*len(ps[a]&ps[b])/len(ps[a]),1)] for a in circ for b in circ if a<b],
                   columns=["A","B","shared_circRNA_ids","shared_pairs","pct_of_A_pairs_in_B"]); display(t)
    print("Example circRNA ids:",{k:datasets[k].rna_ids[:2] for k in circ})

# Do miRNAs that are popular on the circRNA side also have many lncRNA partners?
if circ and lnc:
    def degs(keys): 
        c=pd.Series(dtype=float)
        for k in keys:
            d=datasets[k]; c=c.add(pd.Series(d.A.sum(1),index=[norm_mirna(m) for m in d.mi_ids]).groupby(level=0).sum(),fill_value=0)
        return c
    dc, dl = degs(circ), degs(lnc[:1]); sh=dc.index.intersection(dl.index)
    if len(sh)>5:
        rho,p=spearmanr(dc[sh],dl[sh]); print(f"Spearman between miRNA degree on circRNA side and on {lnc[0]}: rho={rho:.2f} (p={p:.1e}, n={len(sh)} shared miRNAs)")
        plt.figure(figsize=(5,4.5)); plt.scatter(dc[sh]+1,dl[sh]+1,s=12); plt.xscale("log"); plt.yscale("log"); plt.xlabel("miRNA degree (circRNA data)"); plt.ylabel(f"miRNA degree ({lnc[0]})"); plt.title(f"rho={rho:.2f}"); plt.tight_layout(); plt.savefig(f"{OUT}/fig_mirna_degree_transfer.png"); plt.show()

## 8. Unified (merged) positive-pair table
One long table of all known interactions with harmonised miRNA IDs and an RNA-type column. Negatives are generated later per experiment (they depend on the split strategy).

In [ ]:
parts=[]
for k,d in datasets.items():
    p=d.pairs
    parts.append(pd.DataFrame({"source":k,"rna_type":d.rna_type,"rna_id":[d.rna_ids[r] for r in p[:,1]],"mirna_raw":[d.mi_ids[m] for m in p[:,0]],"label":1}))
U=pd.concat(parts,ignore_index=True); U["mirna_norm"]=U.mirna_raw.map(norm_mirna); U["mirna_family"]=U.mirna_raw.map(fam_mirna)
U["pair_key"]=U.rna_type+"|"+U.rna_id.str.lower()+"|"+U.mirna_norm
print("rows:",len(U),"| unique pairs:",U.pair_key.nunique(),"| duplicated across sources:",len(U)-U.pair_key.nunique())
display(U.groupby(["source","rna_type"]).agg(pairs=("label","size"),rnas=("rna_id","nunique"),mirnas=("mirna_norm","nunique")))
U.to_csv(f"{OUT}/unified_positive_pairs.csv",index=False); print("saved", f"{OUT}/unified_positive_pairs.csv")

## 9. Summary table for the decision

In [ ]:
sm=S[["rna_type","n_RNA","n_miRNA","positives","density_pct","RNA_gini","miRNA_gini","top10pct_RNA_edge_share"]].join(
    PB.pivot(index="dataset",columns="negatives",values="AUC_both").add_prefix("popAUC_"))
display(sm.round(3)); sm.to_csv(f"{OUT}/decision_summary.csv")
import shutil; shutil.make_archive("analysis_output","zip",OUT); print("Saved all tables/figures ->", OUT, "and analysis_output.zip")

### How to read the results (decision guide)
| Check | Good for our plan | Bad sign / fallback |
|---|---|---|
| `popAUC_random` | Well below the published ~0.95 (e.g. < 0.8) | Close to 0.95 → published scores are mostly popularity; **an evaluation-critique paper becomes the main contribution** |
| `popAUC_degree` | ≈ 0.5 (shortcut removed) | Still high → degree-matched negatives are not strict enough |
| miRNA overlap (circ vs lnc) | ≥ ~100 shared miRNAs (exact), or a high family-level share | Tiny overlap → drop cross-type transfer; do cold-start benchmark on circRNA data only |
| Shared pairs between D1/D2/D3 | Low → they are independent, safe for external tests | High → cross-dataset test leaks; merge and re-split instead |
| miRNA degree correlation (circ vs lnc) | Weak → miRNA popularity does not simply transfer | Strong → any gain from cross-type transfer may just be popularity |
| Dataset size (DeepLMI ≈ 1k pairs) | — | Small → lncRNA data is a test/transfer set, not the main training set |